# Z4 Z2 TRS cluster chain 100 site DMRG

Created 02/04/2026

Consider a chain of qudits alternating between $d=2$ and $d=4$. Define $$H_{clu} = -\sum Z_4^2 X_2 Z_4^2 -\sum Z_2 X_4 Z_2$$, and $$H_0 = -\sum Y_2 - \sum Z_4$$. Finally define $$H(t) = t H_{clu} + (1-t) H_0$$ which interpolates between the cluster state and product state Hamiltoniains.

Note that $H(t)$ is invariant with respect to $X_4$ on all $d=4$ sites and $X_2 K$ where $X_2$ is taken over all $d=2$ sites and $K$ refers to complex conjugation.

Thus $H(t)$ defines a $Z_4 \times Z_2^T$ SPT. $H^2(G, U(1)) \cong Z_2 \times Z_2$ in this case, with each factor describing $T^2=\pm 1$ and the mixed linear/anti-linear phase. $H(0)$ corresponds to a trivial SPT, $H(1)$ corresponds to an SPT with $T^2=\pm 1$ but non-trivial linear/anti-linear phase.

# Package imports

In [32]:
from tqdm import tqdm

In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

In [2]:
import tenpy
import tenpy.linalg.np_conserved as npc
from tenpy.algorithms import dmrg
from tenpy.networks.mps import MPS

In [5]:
from tenpy.networks.site import SpinSite, ClockSite
from tenpy.models.lattice import Chain
from tenpy.models.model import CouplingMPOModel, NearestNeighborModel, MPOModel

In [23]:
class AlternatingQuditClusterChain(CouplingMPOModel):
        default_lattice = "Chain"
        force_default_lattice = True

        def init_sites(self, model_params):
            site4 = ClockSite(4, conserve=None)
            spin2 = SpinSite(conserve=None)
            return [spin2, site4], ['2', '4']

        def init_terms(self, model_params):
            # Read off model parameters

            t = model_params.get('interpolation', 0)

            self.add_onsite(2*(1-t), 0, 'Sy')
            self.add_onsite(0.5*(1-t), 1, 'Zphc')

            # 2 factor as Sx = (0.5)*X
            self.add_multi_coupling(
                2*t,
                [
                    ('Z', 0, 1),
                    ('Z', 0, 1),
                    ('Sx', 1, 0),
                    ('Z', 1, 1),
                    ('Z', 1, 1),
                ]
            )

            # 2=4/2 factor, 4 from Sz = (0.5)*Z twice and (1/2) as we have two
            # terms from Hermitian conjugate 
            self.add_multi_coupling(
                2*t,
                [
                    ('Sz', 0, 0),
                    ('X', 0, 1),
                    ('Sz', 1, 0),
                ],
                plus_hc=True
            )

In [24]:
import h5py
from tenpy.tools import hdf5_io

In [25]:
dmrg_params = {
    "mixer": True,
    "max_E_err": 1e-10,
    "trunc_params": {
        "chi_max": 10,
        "svd_min": 1e-10
    },
    "max_sweeps": 10
}

In [36]:
interpolation_values = np.linspace(0, 1, 51)

In [37]:
model=AlternatingQuditClusterChain({'interpolation': t, 'L': num_unit_cells})

In [38]:
num_unit_cells = 100

for t in tqdm(interpolation_values):
        print(f"Commencing interpolation={t} model")
        model=AlternatingQuditClusterChain({'interpolation': t, 'L': num_unit_cells})
        
        psi = MPS.from_lat_product_state(model.lat, [[0, '0'],]*num_unit_cells)
        psi.canonical_form()
    
        eng = dmrg.TwoSiteDMRGEngine(psi, model, dmrg_params)
        e, psi = eng.run()

        print(f"Energy: {e}\n")

        data = {
            "wavefunction": psi,
            "energy": e,
            "paramters": {"interpolation": t}
        }

        t_string = str(int(100*t))
        one_minus_t_string = str(int(100*(1-t)))

        file_name = f'{t_string}_4_2_trs_cluster_chain'
    
        filename = (
            r"../data/4_2_trs_cluster_chain_100_unit_cells/{}"
            .format(file_name)
        )

        filename += ".h5"

        with h5py.File(filename, 'w') as f:

            hdf5_io.save_to_hdf5(f, data)

  0%|                                                                                         | 0/51 [00:00<?, ?it/s]

Commencing interpolation=0.0 model


/home/kcooney/Desktop/repos/spt_numerical_classificiation/code/num_spt_venv_p11/lib/python3.11/site-packages/tenpy/tools/params.py:230: UserWarning: unused options for config TwoSiteDMRGEngine:
['norm_tol', 'norm_tol_final']
  warnings.warn(msg.format(keys=sorted(unused), name=self.name))


Energy: -199.99999999999693



  2%|█▌                                                                               | 1/51 [00:07<06:06,  7.33s/it]

...

 96%|████████████████████████████████████████████████████████████████████████████▊   | 49/51 [17:48<00:50, 25.34s/it]

Commencing interpolation=0.98 model
Energy: -194.08718452187645



 98%|██████████████████████████████████████████████████████████████████████████████▍ | 50/51 [18:30<00:30, 30.43s/it]

Commencing interpolation=1.0 model
Energy: -197.99999999999957



100%|████████████████████████████████████████████████████████████████████████████████| 51/51 [19:15<00:00, 22.65s/it]
